# CircuitLM — Train on Personal Data (Kaggle GPU)

Trains a **hybrid CircuitLM** (integer FSM + neural corrector) on your personal conversations.

**Data:** `/kaggle/input/datasets/zacharymaronek/circuit-lm-personal/all_personal_training.txt`

**Output:** `circuit.json` (FSM) + `corrector.pt` (neural corrector) — download these after training.

**Runtime:** GPU required. P100 recommended over 2xT4 — single-GPU training, P100 is faster.

**Time:** ~20-40 min on P100 depending on data size.

## 1. Clone CircuitLM + Install Dependencies

In [ ]:
# Clone the circuit_lm repo
!git clone https://github.com/toxzak-svg/circuit_lm.git /tmp/circuit_lm

# Install dependencies — ortools for CP-SAT circuit training, msgpack for serialization
%pip install ortools msgpack sentencepiece torch --quiet

import sys
sys.path.insert(0, '/tmp/circuit_lm/src')
print('Setup complete')

## 2. Load Training Data

In [ ]:
import os

# Dataset path Zach specified
DATA_PATH = '/kaggle/input/datasets/zacharymaronek/circuit-lm-personal/all_personal_training.txt'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Dataset not found at {DATA_PATH}\n'
        f'Go to kaggle.com → Datasets → Create → Upload all_personal_training.txt\n'
        f'Name the dataset: zacharymaronek/circuit-lm-personal'
    )

with open(DATA_PATH, encoding='utf-8', errors='replace') as f:
    text = f.read()

lines = [l for l in text.strip().split('\n') if l.strip()]
print(f'Loaded {len(lines)} lines ({len(text)/1024/1024:.1f} MB)')
print(f'First 3 lines preview:')
for l in lines[:3]:
    print(f'  {l[:100]}...' if len(l) > 100 else f'  {l}')

## 3. Tokenizer — Hybrid BPE (4096 vocab)

In [ ]:
import time
from circuit_lm.tokenizer import Tokenizer
from circuit_lm.data import load_sequences

VOCAB_SIZE = 4096

print('Building BPE tokenizer...')
t0 = time.time()
tokenizer = Tokenizer.from_text(
    text,
    vocab_size=VOCAB_SIZE,
    mode='bpe',
    bpe_merges=VOCAB_SIZE,
)
print(f'  vocab={tokenizer.vocab_size}, took {time.time()-t0:.1f}s')

print('Tokenizing...')
t0 = time.time()
sequences = load_sequences(text, tokenizer)
total_tokens = sum(len(s) for s in sequences)
print(f'  {len(sequences)} sequences, {total_tokens:,} tokens, took {time.time()-t0:.1f}s')

## 4. Train FSM Circuit (OR-Tools CP-SAT)

This finds the optimal integer state machine that predicts next tokens. Zero floats — pure integer arithmetic at inference.

In [ ]:
from circuit_lm.train_joint_pda_cpsat import train_joint_pda
import time

# Circuit hyperparameters
STATE_BITS = 6       # 64 states
STACK_DEPTH = 4      # PDA stack depth
STACK_STEPS = 15     # Stack operation budget
TRANS_STEPS = 22     # Transition computation budget
EMIT_STEPS = 23      # Emission computation budget

print(f'Training PDA circuit: {STATE_BITS} bits, stack_depth={STACK_DEPTH}')
t0 = time.time()
circuit = train_joint_pda(
    sequences=sequences,
    vocab_size=tokenizer.vocab_size,
    state_bits=STATE_BITS,
    stack_depth=STACK_DEPTH,
    stack_steps=STACK_STEPS,
    transition_steps=TRANS_STEPS,
    emission_steps=EMIT_STEPS,
)
circuit_time = time.time() - t0
print(f'Circuit trained in {circuit_time:.1f}s')

## 5. Save Circuit + Tokenizer

In [ ]:
from circuit_lm.io import save_model

CIRCUIT_PATH = '/tmp/circuit_pda.json'
save_model(circuit, tokenizer, CIRCUIT_PATH)
print(f'Circuit saved: {os.path.getsize(CIRCUIT_PATH)/1024:.0f} KB')

## 6. Train Neural Corrector

The circuit predicts structural patterns. The corrector learns where the circuit is weak and compensates.

Dual-pass inference: circuit proposes → corrector re-ranks.

In [ ]:
import torch
from hybrid import train_hybrid

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CORRECTOR_PATH = '/tmp/corrector_pda.pt'

print('Training neural corrector...')
t0 = time.time()
corrector = train_hybrid(
    circuit_path=CIRCUIT_PATH,
    data_path=DATA_PATH,
    output_path=CORRECTOR_PATH,
    num_epochs=5,
    batch_size=128,
    circuit_weight=0.5,
    max_examples=80000,
)
corrector_time = time.time() - t0
print(f'Corrector trained in {corrector_time:.1f}s')

## 7. Generate a Test Reply

In [ ]:
from hybrid import HybridModel
from circuit_lm.io import load_model

# Load the trained hybrid
hybrid, tok = HybridModel.load(CIRCUIT_PATH, CORRECTOR_PATH, circuit_weight=0.5)

# Test prompt
prompt = 'The best way to build AI that you can trust is'
prompt_ids = tok.encode(prompt)

# Generate 50 tokens
from hybrid import generate_reply_hybrid
reply_ids = generate_reply_hybrid(hybrid, tok, prompt_ids, max_tokens=50)
reply = tok.decode(reply_ids)

print(f'Prompt: {prompt}')
print(f'Generated: {reply}')

## 8. Download Results

In [ ]:
from IPython.display import FileLink, display

print('=== DOWNLOAD THESE FILES ===')
print()
print('Circuit (FSM):')
display(FileLink(CIRCUIT_PATH))
print()
print('Corrector (neural):')
display(FileLink(CORRECTOR_PATH))
print()
print(f'Total training time: {circuit_time + corrector_time:.1f}s')

# Also save tokenizer config for later use
import json
tokenizer_info = {
    'vocab_size': tokenizer.vocab_size,
    'mode': 'bpe',
}
with open('/tmp/tokenizer_info.json', 'w') as f:
    json.dump(tokenizer_info, f)
print('Tokenizer info:')
display(FileLink('/tmp/tokenizer_info.json'))

## Troubleshooting

**OR-Tools takes too long:** Reduce `TRANS_STEPS` and `EMIT_STEPS` to 15 each.
**Out of memory:** Reduce `max_examples` in corrector training to 40000.
**RuntimeError on save_model:** Make sure `msgpack` is installed.

**Next steps after download:**
1. Copy `circuit_pda.json` + `corrector_pda.pt` to your Starfire project
2. Wire into the Rust inference kernel for local, GPU-free inference
3. Use the evolver to find better training recipes